# 🚀 Sen-32B v2: Hybrid Verification & Inference for Tiny LLM (Qwen3.5-0.8B-Instruct)

Notebook này phục vụ việc **chạy thử nghiệm mô hình siêu nhỏ (Tiny LLM - Qwen3.5-0.8B-Instruct)** trên tập dữ liệu **Task 2 Hybrid (Hard + Soft Constraints)**, kết hợp:
1. **Mô hình học sinh (Student Policy):** `Qwen/Qwen3.5-0.8B-Instruct` sinh phản hồi cho các câu prompt phức tạp.
2. **Giám khảo 1 (Hard Verifier):** Code Python thuần túy chấm các quy tắc hình thức (đếm từ, đếm câu, format, P.S., ngoặc kép...).
3. **Giám khảo 2 (Soft Verifier):** `Gemini 3.5 Flash Lite` (đóng vai trò LLM Judge) đọc `eval_rubric` để chấm điểm ngữ nghĩa, văn phong.
4. **Cơ chế Gated Hybrid Reward (VerIF):** Kết hợp điểm thưởng và phạt thẳng tay về 0 nếu model gian lận hình thức mà không đạt ngữ nghĩa.

> 💡 **Môi trường phù hợp:** Google Colab Free Tier (GPU T4 hoặc CPU).

In [ ]:
# @title 1. Cài đặt các thư viện cần thiết
!pip install -q transformers accelerate torch tqdm pandas

In [ ]:
# @title 2. Thiết lập Google API Key (Dành cho Gemini 3.5 Flash Lite LLM Judge)
import os

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    GOOGLE_API_KEY = None

if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = input("Nhập Google Gemini API Key của bạn (hoặc cài đặt trong Colab Secrets): ").strip()

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
print("✓ API Key đã sẵn sàng!")

In [ ]:
# @title 3. Kết nối Google Drive & Nạp Dataset từ /content/drive/MyDrive/vinfast/RLVR
import os
import json
from google.colab import drive

# 1. Mount Google Drive
print("Đang kết nối tới Google Drive...")
drive.mount('/content/drive')

# 2. Thư mục chứa dataset của bạn trên Drive
RLVR_DIR = "/content/drive/MyDrive/vinfast/RLVR"

# Chọn dataset muốn đánh giá (hỗ trợ cả Seen và Unseen OOD):
DATASET_FILENAME = "task2_hybrid_unseen.json" # @param ["task2_hybrid_alpaca_gpt4.json", "task2_hybrid_gpt4_self_instruct.json", "task2_hybrid_unseen.json"]
DRIVE_DATASET_PATH = os.path.join(RLVR_DIR, DATASET_FILENAME)

print(f"Thư mục làm việc: {RLVR_DIR}")
print(f"File được chọn: {DRIVE_DATASET_PATH}")

# Kiểm tra file
if not os.path.isfile(DRIVE_DATASET_PATH):
    alt_path = DRIVE_DATASET_PATH.replace(".json", ".jsonl")
    if os.path.isfile(alt_path):
        DRIVE_DATASET_PATH = alt_path
    else:
        files_in_dir = os.listdir(RLVR_DIR) if os.path.exists(RLVR_DIR) else []
        raise FileNotFoundError(f"Không tìm thấy {DRIVE_DATASET_PATH}. Các file hiện có trong {RLVR_DIR}: {files_in_dir}")

# 3. Đọc dữ liệu
dataset = []
if DRIVE_DATASET_PATH.endswith(".jsonl"):
    with open(DRIVE_DATASET_PATH, "r", encoding="utf-8") as f:
        dataset = [json.loads(line) for line in f if line.strip()]
else:
    with open(DRIVE_DATASET_PATH, "r", encoding="utf-8") as f:
        dataset = json.load(f)

print(f"✓ Nạp thành công {len(dataset)} mẫu dữ liệu từ: {DRIVE_DATASET_PATH}")


In [ ]:
# @title 4. Engine Code Verifier (Chấm Toàn Diện Hard Constraints: 29 Seen + 24 Unseen OOD)
import re
import json
import string
from collections import Counter

def split_sentences(text):
    clean = text.strip()
    parts = re.split(r'(?<=[.!?])\s+', clean)
    return [p.strip() for p in parts if p.strip()]

def verify_hard_constraint(inst_id, kwargs, response_text):
    text = response_text.strip()
    
    # ================== SEEN POOL CONSTRAINTS (29 Quy tắc) ==================
    # 1. Quotation
    if inst_id == "startend:quotation":
        return (text.startswith('"') and text.endswith('"')) or (text.startswith('“') and text.endswith('”'))
    
    # 2. Number of sentences
    if inst_id == "length_constraints:number_sentences":
        num = kwargs.get("num_sentences", 0)
        rel = kwargs.get("relation", "exactly")
        sentences = split_sentences(text)
        count = len(sentences)
        if rel == "less than": return count < num
        if rel == "at least": return count >= num
        return count == num
        
    # 3. Postscript
    if inst_id == "detectable_content:postscript":
        marker = kwargs.get("postscript_marker", "P.S.")
        lines = [line.strip() for line in text.split('\n') if line.strip()]
        if not lines: return False
        return lines[-1].startswith(marker) or marker in text[-150:]
        
    # 4. Words count
    if inst_id == "length_constraints:number_words":
        words = len(text.split())
        num = kwargs.get("num_words", 0)
        rel = kwargs.get("relation", "less than")
        if rel == "less than": return words < num
        if rel == "at least": return words >= num
        if rel == "range": return kwargs.get("min_words", 0) <= words <= kwargs.get("max_words", 9999)
        return words == num
        
    # 5. Paragraphs count
    if inst_id == "length_constraints:number_paragraphs":
        paras = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
        return len(paras) == kwargs.get("num_paragraphs", 0)
        
    # 6. Bullet lists
    if inst_id == "detectable_format:number_bullet_lists":
        bullets = re.findall(r'(?m)^\s*[*\-]\s+', text)
        return len(bullets) == kwargs.get("num_bullets", 0)
        
    # 7. Numbered list
    if inst_id == "format:numbered_list":
        items = re.findall(r'(?m)^\s*\d+\.\s+', text)
        return len(items) == kwargs.get("num_items", 0)
        
    # 8. JSON Format
    if inst_id == "detectable_format:json_format":
        try:
            clean_j = text.strip().strip('`')
            if clean_j.startswith('json'):
                clean_j = clean_j[4:].strip()
            json.loads(clean_j)
            return True
        except Exception: return False
        
    # 9. Title (<<Title>> or [Title])
    if inst_id == "detectable_format:title":
        return bool(re.search(r'<<[^>]+>>', text) or re.search(r'^\s*\[[^\]]+\]', text))
        
    # 10. Keyword existence
    if inst_id == "keywords:existence":
        kws = kwargs.get("keywords", [])
        lower = text.lower()
        return all(re.search(rf'\b{re.escape(w.lower())}\b', lower) for w in kws)
        
    # 11. Keyword frequency
    if inst_id == "keywords:frequency":
        word = kwargs.get("keyword", "").lower()
        freq = kwargs.get("frequency", 0)
        rel = kwargs.get("relation", "exactly")
        actual = len(re.findall(rf'\b{re.escape(word)}\b', text.lower()))
        if rel == "at least": return actual >= freq
        if rel == "at most": return actual <= freq
        return actual == freq
        
    # 12. Letter frequency
    if inst_id == "keywords:letter_frequency":
        letter = kwargs.get("letter", "").lower()
        freq = kwargs.get("let_frequency", 0)
        rel = kwargs.get("let_relation", "at least")
        actual = text.lower().count(letter)
        if rel == "at least": return actual >= freq
        if rel == "at most": return actual <= freq
        return actual == freq
        
    # 13. Forbidden words
    if inst_id == "keywords:forbidden_words":
        forb = kwargs.get("forbidden_words", [])
        lower = text.lower()
        return not any(re.search(rf'\b{re.escape(w.lower())}\b', lower) for w in forb)
        
    # 14. End checker
    if inst_id == "startend:end_checker":
        phrase = kwargs.get("end_phrase", "").strip()
        return text.strip().endswith(phrase) or phrase in text[-len(phrase)-20:]
        
    # 15. Multiple sections
    if inst_id == "detectable_format:multiple_sections":
        n = kwargs.get("num_sections", 0)
        spliter = kwargs.get("section_spliter", "Section")
        sections = re.findall(rf'(?i){re.escape(spliter)}\s*\d+', text)
        return len(sections) >= n
        
    # 16. Highlighted sections
    if inst_id == "detectable_format:number_highlighted_sections":
        n = kwargs.get("num_highlights", 0)
        highlights = re.findall(r'\*[^*]+\*', text)
        return len(highlights) >= n
        
    # 17. No comma
    if inst_id == "punctuation:no_comma":
        return ',' not in text and '，' not in text
        
    # 18. XML wrapper
    if inst_id == "format:xml_wrapper":
        return text.startswith("<response>") and text.endswith("</response>")
        
    # 19. All caps / Lower
    if inst_id == "change_case:english_capital":
        return text.isupper()
    if inst_id == "change_case:english_lowercase":
        return text.islower()

    # 20. Capital words frequency
    if inst_id == "change_case:capital_word_frequency":
        caps = re.findall(r'\b[A-Z]{2,}\b', text)
        return len(caps) >= kwargs.get("capital_words", 2)

    # 21. Response language
    if inst_id == "language:response_language":
        return True # Fallback chấp nhận

    # 22. Paragraph nth first word
    if inst_id == "length_constraints:nth_paragraph_first_word":
        paras = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
        nth = kwargs.get("nth_paragraph", 1) - 1
        if nth < len(paras):
            w = re.findall(r'\b\w+\b', paras[nth].lower())
            return bool(w and w[0] == kwargs.get("first_word", "").lower())
        return False

    # 23. Number placeholders
    if inst_id == "detectable_content:number_placeholders":
        placeholders = re.findall(r'\[[a-zA-Z0-9_\s]+\]', text)
        return len(placeholders) >= kwargs.get("num_placeholders", 1)

    # 24. Constrained response
    if inst_id == "detectable_format:constrained_response":
        opts = kwargs.get("constrained_responses", [])
        lower = text.lower()
        return any(re.search(rf'\b{re.escape(o.lower())}\b', lower) for o in opts)

    # 25. Two responses
    if inst_id == "combination:two_responses":
        return "******" in text

    # 26. Repeat prompt
    if inst_id == "combination:repeat_prompt":
        return len(text.split('\n')) >= 2

    # 27. Table format
    if inst_id == "detectable_format:table_format":
        lines = [l.strip() for l in text.split('\n') if l.strip()]
        table_lines = [l for l in lines if l.startswith('|') and l.endswith('|')]
        return len(table_lines) >= 2

    # 28. No period
    if inst_id == "punctuation:no_period":
        return '.' not in text and '。' not in text

    # ================== UNSEEN POOL (24 Curated IFBench OOD) ==================
    # 1. Unique words count
    if inst_id == "count:unique_word_count":
        words = re.findall(r'\b\w+\b', text.lower())
        return len(set(words)) >= kwargs.get("num_unique_words", 0)

    # 2. Nested parentheses ((like this))
    if inst_id == "format:parentheses":
        nested = re.findall(r'\(\([^\(\)]+\)\)', text)
        return len(nested) >= kwargs.get("num_nested", 1)

    # 3. Palindromic words
    if inst_id == "words:palindrome":
        palindromes = kwargs.get("palindromes", [])
        lower = text.lower()
        return all(re.search(rf'\b{re.escape(p.lower())}\b', lower) for p in palindromes)

    # 4. Sentence increment
    if inst_id == "sentence:increment":
        sentences = split_sentences(text)
        if len(sentences) < 2: return False
        lens = [len(s.split()) for s in sentences]
        return all(lens[i] < lens[i+1] for i in range(len(lens)-1))

    # 5. Start verb
    if inst_id == "words:start_verb":
        paras = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
        if not paras: return False
        COMMON_VERBS = {'eat', 'exercise', 'get', 'drink', 'sleep', 'maintain', 'prioritize', 'practice', 'focus', 'ensure', 'avoid', 'include', 'engage', 'build', 'create', 'develop', 'run', 'walk', 'read', 'write', 'start', 'begin', 'take', 'make', 'do', 'use', 'stay', 'keep', 'choose', 'aim', 'limit', 'reduce', 'boost', 'protect', 'adopt', 'schedule', 'cultivate', 'foster', 'incorporate', 'strive'}
        for para in paras:
            first_word = re.findall(r'\b\w+\b', para.lower())
            if not first_word: return False
            w = first_word[0]
            if w not in COMMON_VERBS and not (w.endswith('ing') or w.endswith('ed')):
                return False
        return True

    # 6. Punctuation count
    if inst_id == "count:punctuation":
        found = set(c for c in text if c in string.punctuation)
        return len(found) >= kwargs.get("num_punctuation_types", 4)

    # 7. Alphabet loop
    if inst_id == "words:alphabet_loop":
        words = re.findall(r'\b[a-zA-Z]', text.lower())
        if len(words) < 3: return False
        return all((ord(words[i]) - ord('a')) == (ord(words[i-1]) - ord('a') + 1) % 26 for i in range(1, len(words)))

    # 8. Prime lengths
    if inst_id == "words:prime_lengths":
        primes = {2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31}
        words = re.findall(r'\b[a-zA-Z]+\b', text)
        if not words: return False
        return all(len(w) in primes for w in words)

    # 9. Nested quotes
    if inst_id == "format:nested_quotes":
        return bool(re.search(r'("[^"]*'[^']*("[^"]*"|'[^']*')[^']*'[^"]*")|('[^']*"[^"]*('[^']*'|"[^"]*")[^"]*"[^']*')', text))

    # 10. Numbers count
    if inst_id == "count:numbers_count":
        nums = re.findall(r'\b\d+\b', text)
        return len(nums) == kwargs.get("num_numbers", 3)

    # 11. Title case
    if inst_id == "format:title_case":
        words = re.findall(r'\b[a-zA-Z]+\b', text)
        if not words: return False
        cap_count = sum(1 for w in words if w[0].isupper())
        return (cap_count / len(words)) >= 0.75

    # 12. Last word first next
    if inst_id == "sentence:last_word_first_next":
        sentences = split_sentences(text)
        if len(sentences) < 2: return False
        for i in range(len(sentences) - 1):
            w_last = re.findall(r'\b\w+\b', sentences[i].lower())
            w_first = re.findall(r'\b\w+\b', sentences[i+1].lower())
            if not w_last or not w_first or w_last[-1] != w_first[0]:
                return False
        return True

    # 13. No consecutive first letter
    if inst_id == "words:no_consecutive_first_letter":
        words = re.findall(r'\b[a-zA-Z]', text.lower())
        if len(words) < 2: return True
        return all(words[i] != words[i+1] for i in range(len(words) - 1))

    # 14. Limited word repeat
    if inst_id == "words:limited_repeat":
        words = re.findall(r'\b\w+\b', text.lower())
        if not words: return False
        return Counter(words).most_common(1)[0][1] <= kwargs.get("max_repeats", 4)

    # 15. Word count step
    if inst_id == "sentence:word_count_step":
        sentences = split_sentences(text)
        if len(sentences) < 2: return False
        lens = [len(re.findall(r'\b\w+\b', s)) for s in sentences]
        step = kwargs.get("step", 2)
        return all(lens[i+1] - lens[i] == step for i in range(len(lens)-1))

    # 16. Date format list
    if inst_id == "format:date_format_list":
        dates = re.findall(r'\b\d{4}-\d{2}-\d{2}\b', text)
        return len(dates) >= kwargs.get("min_dates", 2)

    # 17. CSV format
    if inst_id == "format:csv_format":
        lines = [l.strip() for l in text.split('\n') if l.strip()]
        if len(lines) < kwargs.get("num_rows", 3): return False
        return all(',' in l for l in lines)

    # 18. Output template
    if inst_id == "format:output_template":
        return "My Answer:" in text and "My Conclusion:" in text and "Future Outlook:" in text

    # 19. Paragraph last first word match
    if inst_id == "words:paragraph_last_first_match":
        paras = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
        if not paras: return False
        for p in paras:
            words = re.findall(r'\b\w+\b', p.lower())
            if len(words) < 2 or words[0] != words[-1]:
                return False
        return True

    # 20. Conjunction count
    if inst_id == "words:conjunction_count":
        conj_list = {'and', 'but', 'for', 'nor', 'or', 'so', 'yet'}
        words = set(re.findall(r'\b\w+\b', text.lower()))
        used = words.intersection(conj_list)
        return len(used) >= kwargs.get("min_conjunctions", 3)

    # 21. Indent stairs
    if inst_id == "format:indent_stairs":
        lines = [l for l in text.split('\n') if l.strip()]
        if len(lines) < 3: return False
        indents = [len(l) - len(l.lstrip(' ')) for l in lines]
        return all(indents[i] < indents[i+1] for i in range(len(indents)-1))

    # 22. Special bullet
    if inst_id == "format:special_bullet":
        lines = [l.strip() for l in text.split('\n') if l.strip()]
        bullet_lines = [l for l in lines if l.startswith('->')]
        return len(bullet_lines) >= 2

    # 23. No whitespace
    if inst_id == "manipulation:no_whitespace":
        return not any(c in string.whitespace for c in text)

    # 24. Symbol end
    if inst_id == "sentence:symbol_end":
        sentences = split_sentences(text)
        if not sentences: return False
        return all(s.endswith('!*') or '!*' in s[-5:] for s in sentences)

    # Default fallback
    return True


In [ ]:
# @title 5. Engine Soft Verifier (Gemini 3.5 Flash Lite LLM-as-a-Judge)
import urllib.request
import time

def evaluate_soft_constraint_gemini(response_text, soft_constraint, api_key, model="gemini-3.5-flash-lite"):
    c_desc = soft_constraint.get("description", "")
    rubric = soft_constraint.get("eval_rubric", "")
    
    judge_prompt = f'''You are an impartial, expert evaluation judge assessing an LLM's response quality for instruction following.

Criterion to Evaluate:
{c_desc}

Evaluation Rubric / Scoring Standard:
{rubric}

Candidate Response to Assess:
"""
{response_text}
"""

Decision Task:
Determine strictly whether the response satisfies the rubric criteria above.
Return ONLY a JSON object:
{{
  "reasoning": "<1-2 concise sentences explaining the rationale>",
  "score": 1 or 0
}}
'''
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={api_key}"
    payload = {
        "contents": [{"parts": [{"text": judge_prompt}]}],
        "generationConfig": {"temperature": 0.1, "responseMimeType": "application/json"}
    }
    data = json.dumps(payload).encode("utf-8")
    
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"}, method="POST")
            with urllib.request.urlopen(req, timeout=20) as resp:
                res = json.loads(resp.read().decode("utf-8"))
                raw_text = res["candidates"][0]["content"]["parts"][0]["text"]
                parsed = json.loads(raw_text)
                return int(parsed.get("score", 0)), parsed.get("reasoning", "")
        except Exception as e:
            time.sleep(2.0 * (attempt + 1))
            
    return 0, "Judge call failed"

print("✓ Engine Soft Verifier (Gemini 3.5 Flash Lite) đã sẵn sàng!")

In [ ]:
# @title 6. Tải Tiny LLM (Mặc định: Qwen/Qwen2.5-0.5B-Instruct)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Chọn mô hình siêu nhỏ (Tiny LLM) từ Hugging Face:
# Lưu ý: Alibaba Qwen hiện có các bản 0.5B và 1.5B (chưa có bản 0.8B hay thế hệ Qwen 3.5)
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct" # @param ["Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct", "meta-llama/Llama-3.2-1B-Instruct"]
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang tải mô hình {MODEL_ID} lên thiết bị: {device}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

if device == "cpu":
    model = model.to("cpu")

print(f"✓ Tải mô hình {MODEL_ID} thành công và sẵn sàng chạy!")


In [ ]:
# @title 7. Chạy Inference & Chấm điểm Chuẩn (Hard-Priority Multiplicative Gating)
from tqdm import tqdm
import pandas as pd

# Chọn số lượng mẫu muốn test (-1 hoặc 0 để chạy toàn bộ dataset):
NUM_TEST_SAMPLES = 10 # @param {type:"integer"}
test_subset = dataset[:NUM_TEST_SAMPLES] if (NUM_TEST_SAMPLES > 0 and NUM_TEST_SAMPLES < len(dataset)) else dataset
print(f"-> Sẽ chạy đánh giá trên {len(test_subset)} mẫu dữ liệu.")

eval_results = []

for idx, item in enumerate(tqdm(test_subset, desc="Đang chạy Inference & Chấm thi")):
    prompt = item["prompt"]
    key = item.get("key", f"sample_{idx}")
    
    # 1. Sinh câu trả lời từ Tiny LLM
    messages = [{"role": "user", "content": prompt}]
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text_input], return_tensors="pt").to(device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    qwen_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    # 2. Chấm điểm Hard Constraints (Trọng tâm tối thượng của IFEval/RLVR)
    gt_list = item.get("hard_constraints", {}).get("ground_truth", [])
    hard_passed = 0
    hard_total = 0
    if gt_list:
        for gt in gt_list:
            ids = gt.get("instruction_id", [])
            kwargs_list = gt.get("kwargs", [])
            for c_id, kw in zip(ids, kwargs_list):
                hard_total += 1
                if verify_hard_constraint(c_id, kw, qwen_response):
                    hard_passed += 1
    
    r_hard = (hard_passed / hard_total) if hard_total > 0 else 1.0
    
    # 3. Chấm điểm Soft Constraints (qua Gemini 3.5 Flash Lite làm Semantic Quality Filter)
    soft_list = item.get("soft_constraints", {}).get("constraints", [])
    soft_passed = 0
    soft_total = len(soft_list)
    soft_reasonings = []
    
    # Tối ưu chi phí & tốc độ: Nếu r_hard == 0.0 (sai toàn bộ luật cứng), coi như trượt hoàn toàn
    if r_hard == 0.0 and soft_total > 0:
        r_soft = 0.0
        r_total = 0.0
        gate_status = "FAILED_HARD_ZERO (Vi phạm toàn bộ quy tắc cứng)"
        soft_reasonings.append("Bỏ qua chấm Soft vì Hard Rules = 0.0")
    else:
        for sc in soft_list:
            score, reason = evaluate_soft_constraint_gemini(qwen_response, sc, api_key=GOOGLE_API_KEY)
            soft_passed += score
            sc_id = sc.get('id', 'soft')
            soft_reasonings.append(f"[{sc_id}]: {score} ({reason})")
            time.sleep(1.0) # Tránh 429
            
        r_soft = (soft_passed / soft_total) if soft_total > 0 else 1.0
        
        # 4. Cơ chế Tính Reward Chuẩn (Multiplicative Gating):
        # Đảm bảo: Nếu R_hard = 0 -> R_total LUÔN = 0.0!
        if r_hard == 0.0:
            r_total = 0.0
            gate_status = "FAILED_HARD_ZERO"
        elif r_soft < 0.5:
            r_total = 0.0
            gate_status = "FAILED_OFF_TOPIC (Reward Hacking blocked!)"
        else:
            # Kết hợp với trọng số ưu tiên Hard (75% Hard, 25% Soft)
            r_total = round(0.75 * r_hard + 0.25 * r_soft, 2)
            gate_status = "PASSED"
        
    eval_results.append({
        "key": key,
        "base_instruction": item.get("base_instruction", "")[:60],
        "qwen_response": qwen_response,
        "hard_score": round(r_hard, 2),
        "soft_score": round(r_soft, 2),
        "total_reward": round(r_total, 2),
        "gate_status": gate_status,
        "soft_eval_log": "; ".join(soft_reasonings)
    })

print("\n✓ ĐÃ HOÀN TẤT ĐÁNH GIÁ!")


In [ ]:
# @title 8. Bảng Tổng Kết Kết Quả & Thống Kê
df = pd.DataFrame(eval_results)
print(f"--- BÁO CÁO ĐO LƯỜNG TRÊN {len(df)} MẪU ---")
print(f"• Điểm Hard Rules TB: {df['hard_score'].mean():.2%}")
print(f"• Điểm Soft Rules TB: {df['soft_score'].mean():.2%}")
print(f"• Điểm Total Reward TB (Sau Gate): {df['total_reward'].mean():.2%}")
print(f"• Tỷ lệ bị phạt Gate về 0: {(df['total_reward'] == 0.0).mean():.2%}")

# Hiển thị 5 kết quả mẫu
df[['key', 'hard_score', 'soft_score', 'total_reward', 'gate_status']].head(10)

In [ ]:
# @title 9. Xuất & Lưu Kết Quả vào /content/drive/MyDrive/vinfast/RLVR
import os

RLVR_DIR = "/content/drive/MyDrive/vinfast/RLVR"
os.makedirs(RLVR_DIR, exist_ok=True)

# Tên file kết quả dựa trên dataset đang chạy
base_name = DATASET_FILENAME.replace(".jsonl", "").replace(".json", "")
drive_csv = os.path.join(RLVR_DIR, f"{base_name}_qwen_verif_results.csv")
drive_json = os.path.join(RLVR_DIR, f"{base_name}_qwen_verif_results.json")

# Lưu trực tiếp vào Google Drive
df.to_csv(drive_csv, index=False, encoding="utf-8-sig")
with open(drive_json, "w", encoding="utf-8") as f:
    json.dump(eval_results, f, ensure_ascii=False, indent=2)

print(f"✓ Đã lưu thành công kết quả vào thư mục Drive:")
print(f"  • CSV:  {drive_csv}")
print(f"  • JSON: {drive_json}")
